In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
import re
import time
import xgboost as xgb
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler, OneHotEncoder, QuantileTransformer
from sklearn.datasets import fetch_california_housing
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier


# Understanding and Getting The Data

In [ ]:
train_data = pd.read_csv('/kaggle/input/titanic/train.csv')
display(train_data.head())
print(train_data.info())
print(train_data.isnull().sum())
print("\nJumlah Value per Kolom (Manual):")
for kolom in train_data.drop(['PassengerId', 'Name', 'Ticket', 'Cabin','Age', 'Ticket', 'Fare'], axis=1).columns:
    print(f"\n--- Kolom: {kolom} ---")
    print(train_data[kolom].value_counts())
sns.histplot(data=train_data, x='Fare', kde=True, bins=12)
plt.title('Fare Histogram with KDE')
plt.show()

# Feature Engineering
1. Add SibSp and Parch to get their full family
2. Fill missing age from their title if they have one

For names:
1. Split each name to get their titles
2. Confirm their sex with their titles (just in case)
3. Get their last name so that we could concat their family

In [ ]:
def FeatureEngineering(df):
    df['Title'] = df['Name'].str.extract(r', ([A-Za-z]+)\.', expand=False)
    
    # Replace simple female title with Miss or Mrs
    df['Title'] = df['Title'].replace(['Mlle', 'Ms'], 'Miss')
    df['Title'] = df['Title'].replace(['Mme'], 'Mrs')
    # Replace unique title with 'Rare'
    df['Title'] = df['Title'].replace(['Dr', 'Rev', 'Col', 'Major', 'Lady','Sir', 'Capt', 'Don', 'Jonkheer', 'Dona'], 'Rare')
    df['Title'] = df['Title'].fillna('Rare')
    df.drop('Name', axis = 1, inplace=True)
    
    # Checking the value
    print(df['Title'].value_counts())
    print("\n\n")
    
    # Fill missing age
    title_age_median = train_data.groupby('Title')['Age'].median()
    def FillAge(age, title):
        if pd.isnull(age):
            return title_age_median[title]
        else:
            return age
        
    df['Age'] = df.apply(lambda row: FillAge(row['Age'], row['Title']), axis=1)
    
    # Add SibSp with Parch
    df['FamilyMembers'] = df['SibSp'] + df['Parch']
    df.drop(['SibSp', 'Parch'], axis=1, inplace=True)
    df['IsAlone'] = (df['FamilyMembers'] == 0).astype(int)

    # Deck for Cabin
    df['Cabin'] = df['Cabin'].notna().astype(int)
    def get_ticketPrefix(ticket):
        ticket = re.sub(r'[\./]', '', ticket)
        parts = ticket.split(' ')
        if(len(parts) > 0 and not parts[0].isdigit()):
            return parts[0]
        else:
            return 'NUM'

    df['Ticket'] = df['Ticket'].apply(get_ticketPrefix)
    
    print("\n\n---Current Datasets---\n\n")
    print(df)

FeatureEngineering(train_data)

# Modelling and Pipelining
We will use RandomForestClassifire, StandardScaler (for nnumerical features), and One-Hot-Encoding (for categorial features)

In [ ]:

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
    ]
)
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

numerical_features = ['Age', 'Fare', 'FamilyMembers']
categorial_features = ['Pclass', 'Sex', 'Embarked', 'Title', 'IsAlone', 'Cabin', 'Ticket']
binary_features = ['IsAlone', 'Cabin']

preprocessor = ColumnTransformer(
    transformers = [
        ('num', num_transformer, numerical_features),
        ('cat', cat_transformer, categorial_features),
        ('bin', 'passthrough', binary_features),
    ]
)



# Create the model pipeline
model_pipeline =   Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', xgb.XGBClassifier(random_state=42, 
                                     use_label_encoder=False, 
                                     eval_metric='logloss'))  
])

parameter_dist ={
    # Jumlah pohon (estimators)
    'classifier__n_estimators': [100, 200, 300, 500, 1000],
    
    # Seberapa cepat model belajar (learning rate)
    'classifier__learning_rate': [0.01, 0.05, 0.1, 0.2],
    
    # Kedalaman maksimum pohon (seperti di RF)
    'classifier__max_depth': [3, 4, 5, 6, 8],
    
    # % sampel yang digunakan untuk setiap pohon (mencegah overfitting)
    'classifier__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    
    # % fitur yang digunakan untuk setiap pohon (mencegah overfitting)
    'classifier__colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    
    # Regularisasi (semakin tinggi, semakin konservatif modelnya)
    'classifier__gamma': [0, 0.1, 0.25, 0.5, 1.0]
}

random_search = RandomizedSearchCV(
    estimator=model_pipeline,
    param_distributions=parameter_dist,
    n_iter=20, 
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=2 
)

# Splitting the train data and training it with the model

In [ ]:
X = pd.concat([train_data[numerical_features], train_data[categorial_features]], axis = 1)
y = train_data['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Memulai Hyperparameter Tuning...")
start_time = time.time()
random_search.fit(X_train, y_train)
end_time = time.time()
print(f"Tuning selesai dalam {end_time - start_time:.2f} detik.")


print("\n--- Hasil Tuning ---")
print(f"Parameter Terbaik Ditemukan: {random_search.best_params_}")
prediction = random_search.predict(X_test)


print("\n--- Full Pipeline Results ---")
print("Test data (raw):\n")
display(X_test)
print("Predictions:")
display(pd.DataFrame(prediction))
print("Actual:")
display(y_test)



results_df = pd.DataFrame({
    'Actual (y_test)': y_test,
    'Predicted': prediction
})
display(results_df)

# Cetak laporan klasifikasi
print("\n--- Classification Report ---")
print(classification_report(y_test, prediction))

# Cetak confusion matrix
print("\n--- Confusion Matrix ---")
print(confusion_matrix(y_test, prediction))

# Outputing and Finalizing The Model

In [ ]:
best_model = random_search.best_estimator_

test_df = pd.read_csv('/kaggle/input/titanic/test.csv')
FeatureEngineering(test_df)
print("Data preprocessing is done.")

X_final_test = pd.concat([test_df[numerical_features], test_df[categorial_features]], axis=1)

print("Making predictions for X_final_test...")
final_predictions = best_model.predict(X_final_test)

# Make a submission file
submission_df = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": final_predictions
})

# 6. Simpan File
submission_df.to_csv('submission.csv', index=False)

print("\n--- Finished ---")
print("'submission.csv'.")

# Tampilkan 5 baris pertama dari file submission
print("\nExample of submission.csv file:")
display(submission_df.head())